# Robust sensitivity analysis (Γ = 26)

This notebook only **plots** results. The solves are done by `src/run_sweeps.jl`, which writes
`results/sensitivity/runs_<tag>.csv` (one row per run), `sites_<tag>.csv` (chosen sites) and
`rounds_<tag>.csv` (cutting-plane rounds), one set per parallel worker; this notebook stacks them. Run from the project root:

```
julia --project=@v1.11 -t auto src/run_sweeps.jl
```

Every run is the **robust** model of README2 §3.2 with uncertainty budget Γ = 26 block groups.

**Baseline:** grid spacing Δ = 500 m, K = 10 new pantries, `J_MAX` = 2, `N_NEW` = 2,
capacity 900 people/month for every pantry, q = 5,000 m, D_max = 3 km, demand = people below
1.85 × poverty.

| Sweep | What varies | Held fixed |
|---|---|---|
| Grid size × K | Δ ∈ {1000, 500, 250} m, K ∈ {0, 2, …, 20} | baseline caps and capacity |
| Capacity | capacity ∈ {600, 900, 1,200} (existing and new alike) × K ∈ {0, 2, …, 20} | Δ = 500 m |

- `J_MAX`: most pantries in one block group, existing + new.
- `N_NEW`: most **new** pantries in one block group.
- The `J_MAX` and `N_NEW` cap sweeps were solved but are not plotted: the caps never bound
  (no block group got more than one new pantry). Their runs are still in the CSVs.

**Metrics** (all for the sites the robust model chose):

| Column | Meaning |
|---|---|
| `worst_cost` | worst-case objective found: person-metres travelled + q × people unserved |
| `served_worst`, `avg_dist_worst_m` | people served and average trip (m) under that worst-case demand |
| `served_nominal`, `avg_dist_nominal_m` | the same under the census estimates |
| `nominal_sites_worst_cost` | cost of the **nominal** model's sites under the same worst-case demand |
| `sites_changed_vs_nominal` | how many robust sites differ from the nominal model's sites |

In [ ]:
using CSV, DataFrames, Plots
RES = joinpath("..", "results", "sensitivity")
# One runs*.csv per sweep worker; stack them
read_all(prefix) = vcat([CSV.read(joinpath(RES, f), DataFrame; types = Dict(:J_MAX => String, :N_NEW => String))
                         for f in readdir(RES) if startswith(f, prefix) && endswith(f, ".csv")]...; cols = :union)
runs = unique(read_all("runs"), :run_key)
runs.cost_M = runs.worst_cost ./ 1e6                       # million person-metres
println(nrow(runs), " runs; all converged: ", all(runs.converged), "; max gap: ",
        round(100 * maximum(runs.gap), digits = 2), " %")
# Fixed colours: grid spacing = blue / green / purple, capacity = red / black / gold
GRID_COLORS = Dict(1000 => "#1f77b4", 500 => "#2ca02c", 250 => "#9467bd")
CAP_COLORS  = Dict(600.0 => "#d62728", 900.0 => "#000000", 1200.0 => "#e6ab02")
base(df) = df[(df.J_MAX .== "2") .& (df.N_NEW .== "2") .& (df.c_exist .== 900), :]
default(size = (750, 480), left_margin = 8Plots.mm, bottom_margin = 6Plots.mm,
        legend = :topright, markersize = 4, linewidth = 2)

## 1. Grid size and K

Worst-case objective against K, one line per grid spacing.

In [ ]:
kg = sort(base(runs), [:delta, :K])
fig1 = plot(xlabel = "K (max new pantries)", ylabel = "Worst-case objective (million person-m)",
            title = "Objective vs K, by grid spacing", xticks = 0:2:20)
for d in sort(unique(kg.delta); rev = true)
    s = kg[kg.delta .== d, :]
    plot!(fig1, s.K, s.cost_M; marker = :o, color = GRID_COLORS[d], label = "Δ = $d m")
end
savefig(fig1, joinpath(RES, "grid_K.png")); fig1

In [ ]:
# Objective at K = 10 by grid spacing, relative to the finest grid
k10 = sort(kg[kg.K .== 10, [:delta, :cost_M, :served_worst, :avg_dist_worst_m, :solve_sec]], :delta; rev = true)
k10.cost_vs_finest_pct = round.(100 .* (k10.cost_M ./ minimum(k10.cost_M) .- 1), digits = 2)
k10

## 2. Capacity × K (Δ = 500 m)

Capacity is changed for existing and new pantries together.

In [ ]:
cap = sort(runs[(runs.delta .== 500) .& (runs.J_MAX .== "2") .& (runs.N_NEW .== "2"), :], [:c_exist, :K])
fig3 = plot(xlabel = "K (max new pantries)", ylabel = "Worst-case objective (million person-m)",
            title = "Objective vs K, by pantry capacity (Δ = 500 m)", xticks = 0:2:20, legend = :bottomleft)
for c in sort(unique(cap.c_exist))
    s = cap[cap.c_exist .== c, :]
    plot!(fig3, s.K, s.cost_M; marker = :o, color = CAP_COLORS[c], label = "capacity $(Int(c)) people/month")
end
savefig(fig3, joinpath(RES, "capacity_K.png")); fig3

## 3. The two parts of the objective

The objective = **travel** (person-metres: people served × their trip length) + **unserved penalty**
(q = 5,000 m × people left unserved), both under the worst-case demand. Left panels: travel. Right
panels: penalty. Note the very different y-axis scales.

The travel part is small, so the 0.5 % solver gap (about 3 million person-m) is a large share of it.
Wiggles in the travel lines are within that tolerance.

In [ ]:
runs.travel_M  = runs.served_worst .* runs.avg_dist_worst_m ./ 1e6
runs.penalty_M = runs.q .* (runs.demand_worst .- runs.served_worst) ./ 1e6
println("travel + penalty matches objective: ", all(isapprox.(runs.travel_M .+ runs.penalty_M, runs.cost_M; rtol = 1e-6)))

function parts_plot(df, groupcol, label, colors, title)
    a = plot(xlabel = "K (max new pantries)", ylabel = "Travel (million person-m)", xticks = 0:2:20, legend = false)
    b = plot(xlabel = "K (max new pantries)", ylabel = "Unserved penalty (million person-m)", xticks = 0:2:20,
             legend = :bottomleft)
    for g in sort(unique(df[!, groupcol]); rev = groupcol == :delta)
        s = sort(df[df[!, groupcol] .== g, :], :K)
        plot!(a, s.K, s.travel_M; marker = :o, color = colors[g], label = label(g))
        plot!(b, s.K, s.penalty_M; marker = :o, color = colors[g], label = label(g))
    end
    plot(a, b; layout = (1, 2), size = (1300, 480), plot_title = title,
         left_margin = 10Plots.mm, bottom_margin = 8Plots.mm)
end

fig4 = parts_plot(base(runs), :delta, d -> "Δ = $d m", GRID_COLORS, "Objective parts vs K, by grid spacing")
savefig(fig4, joinpath(RES, "parts_grid_K.png")); fig4

In [ ]:
capruns = runs[(runs.delta .== 500) .& (runs.J_MAX .== "2") .& (runs.N_NEW .== "2"), :]
fig5 = parts_plot(capruns, :c_exist, c -> "capacity $(Int(c))", CAP_COLORS, "Objective parts vs K, by pantry capacity (Δ = 500 m)")
savefig(fig5, joinpath(RES, "parts_capacity_K.png")); fig5

## 4. Average trip distance

Same layout as section 3, with the travel panel replaced by the average distance (m) travelled by the
people served, under the worst-case demand (travel ÷ people served). Right panels: unserved penalty, as before.
Same caveat as the travel panel: the 0.5 % solver gap can move this by several metres, so wiggles
(e.g. the spikes at small K on the 500 m and 250 m grids) are within solver tolerance.

In [ ]:
function avgdist_plot(df, groupcol, label, colors, title)
    a = plot(xlabel = "K (max new pantries)", ylabel = "Average trip (m, worst case)", xticks = 0:2:20, legend = false)
    b = plot(xlabel = "K (max new pantries)", ylabel = "Unserved penalty (million person-m)", xticks = 0:2:20,
             legend = :bottomleft)
    for g in sort(unique(df[!, groupcol]); rev = groupcol == :delta)
        s = sort(df[df[!, groupcol] .== g, :], :K)
        plot!(a, s.K, s.avg_dist_worst_m; marker = :o, color = colors[g], label = label(g))
        plot!(b, s.K, s.penalty_M; marker = :o, color = colors[g], label = label(g))
    end
    plot(a, b; layout = (1, 2), size = (1300, 480), plot_title = title,
         left_margin = 10Plots.mm, bottom_margin = 8Plots.mm)
end
fig6 = avgdist_plot(base(runs), :delta, d -> "Δ = $d m", GRID_COLORS, "Average trip and unserved penalty vs K, by grid spacing")
savefig(fig6, joinpath(RES, "avg_dist_grid_K.png")); fig6

In [ ]:
fig7 = avgdist_plot(capruns, :c_exist, c -> "capacity $(Int(c))", CAP_COLORS, "Average trip and unserved penalty vs K, by pantry capacity (Δ = 500 m)")
savefig(fig7, joinpath(RES, "avg_dist_capacity_K.png")); fig7

## 5. Does robustness change the sites?

`benefit_pct`: how much worse the nominal model's sites do than the robust sites under the
worst-case demand the adversary found. 0 means robustness changed nothing.

In [ ]:
runs.benefit_pct = 100 .* (runs.nominal_sites_worst_cost ./ runs.worst_cost .- 1)
runs.price_pct   = 100 .* (runs.worst_cost ./ runs.nominal_cost .- 1)   # worst case vs estimate, robust sites
println("runs where robust sites differ from nominal: ", count(>(0), runs.sites_changed_vs_nominal), " of ", nrow(runs))
sort(runs[:, [:run_key, :sites_changed_vs_nominal, :benefit_pct, :price_pct, :rounds, :solve_sec]],
     :sites_changed_vs_nominal; rev = true)